In [5]:
GITHUB_USERNAME = "abdelrahmanelenany"
REPO_NAME = "stock-prediction" # replace if your repo has a different name

# Clean up before cloning to handle session restarts
!rm -rf {REPO_NAME}
!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git

# Change working directory so all relative paths work just like locally
%cd {REPO_NAME}

Cloning into 'stock-prediction'...
remote: Enumerating objects: 217, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 217 (delta 53), reused 124 (delta 32), pack-reused 67 (from 1)
Receiving objects: 100% (217/217), 464.38 KiB | 11.61 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/stock-prediction/stock-prediction


In [6]:
import glob

# Search for Python files recursively
py_files = glob.glob("**/*.py", recursive=True)

# Replace the Apple Silicon MPS check with a CUDA check that falls back to MPS/CPU
old_logic_single = "torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')"
old_logic_multi = "device = (\n    torch.device('mps') if torch.backends.mps.is_available()\n    else torch.device('cpu')\n)"

new_logic_single = "torch.device('cuda') if torch.cuda.is_available() else torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')"
new_logic_multi = "device = (\n    torch.device('cuda') if torch.cuda.is_available()\n    else torch.device('mps') if torch.backends.mps.is_available()\n    else torch.device('cpu')\n)"

patched_files = 0
for filepath in py_files:
    with open(filepath, "r") as f:
        content = f.read()

    modified = False
    if old_logic_multi in content:
        content = content.replace(old_logic_multi, new_logic_multi)
        modified = True

    if old_logic_single in content:
        content = content.replace(old_logic_single, new_logic_single)
        modified = True

    if modified:
        with open(filepath, "w") as f:
            f.write(content)
        print(f"✅ Patched CUDA support into: {filepath}")
        patched_files += 1

if patched_files == 0:
    print("No changes made. (Codebase may already support CUDA or the exact check was not found).")

✅ Patched CUDA support into: main.py
✅ Patched CUDA support into: experiments/lstm_lr_sweep.py


In [7]:
import torch

print("--- Hardware Verification ---")
if torch.cuda.is_available():
    print(f"🚀 Success! Using Cloud GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: CPU detected. LSTMs will train very slowly. Please enable the GPU in your environment settings.")

# Install yfinance and other dependencies
!pip install -q -r requirements.txt
print("Dependencies installed successfully.")

--- Hardware Verification ---
🚀 Success! Using Cloud GPU: Tesla T4
Dependencies installed successfully.


In [8]:
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("✅ Ensured output directories exist.")

✅ Ensured output directories exist.


In [ ]:
import sys
# A useful magic trick! This ensures any realtime print/logging is outputted to the cell smoothly
!python -u main.py

Using device: cuda
BACKTEST — 5 MODELS × N FOLDS
LSTM Tuning: DISABLED
Scaler Type: standard
[*********************100%***********************]  105 of 105 completed
Raw data saved: 2515 rows x 525 columns
Removed 2 zero-volume rows
Dropped 0 rows with missing Close
      Date Ticker      Close  Return_1d
2020-03-16   AMAT  38.369556  -0.203576
2016-04-22    AMD   3.990000   0.522901
2017-05-02    AMD  10.320000  -0.242291
2024-12-13   AVGO 221.726364   0.244326
2020-03-24    AXP  77.725426   0.218823
2020-11-09    AXP 110.094193   0.213879
2020-03-16     BA 129.610001  -0.238484
2020-03-24     BA 127.680000   0.208862
2020-03-25     BA 158.729996   0.243186
2018-04-26    CMG   8.450000   0.244404
2020-03-09    COP  27.338779  -0.248401
2020-03-24    COP  24.666687   0.252139
2020-08-26    CRM 269.050934   0.260449
2020-03-18    CVX  42.203594  -0.221248
2020-03-24    CVX  51.019966   0.227407
2016-07-05    DHR  67.867836   0.612184
2020-03-09    EOG  28.486580  -0.320072
2022-09-16   

In [ ]:
import shutil
from IPython.display import FileLink

# Zip up all outputs, processed datasets, and generated reports
!zip -q -r result_exports.zip reports/ outputs/ data/processed/

print("Files compressed! Use the file explorer on the left to download 'result_exports.zip', or click the link below:")
FileLink('result_exports.zip')